In [2]:
import numpy as np
import pandas as pd

# Retirement Subsidy to Grade 4 & 5 Diesel Transportations

[Korea plans to subsidize the early retirement of diesel cars of grade 4 and grade 5 and that of gasoline cars of grade 5.](https://www.mecar.or.kr/lpm/info/oldCarEarlyScrapping.do) In 2019.6, the number of enrolled diesel cars was 9,989,629. That of gasoline was 11,140,964. [That of grade 3 diesel cars was 6,200,340. That of grade 4 diesel cars was 1,350,352. And that of grade 5 was 2,438,937.](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156338348#pressRelease) That of grade 5 gasoline cars was 31,116. [By the Korean government, the retirement rate for grade 4 diesel cars increased by 9.6%p.](https://www.hankyung.com/article/202402181041i) This effect is assumed to be applied for grade 4 and grade 5 diesel and grade 5 gasoline. Refer to 2022 stats [here](https://ctis.re.kr/en/selectBbsNttView.do?key=1574&bbsNo=312&nttNo=1129021&searchCtgry=&searchCnd=all&searchKrwd=&pageIndex=5&searchBbsType=&chgPage=)

* 정책은 2024년부터 시행된다고 가정
* Enhanced: 2025년 이후 3등급도 폐차

In [13]:
yearly_effect = (0.141 - 0.045) * (1160239 + 1140167) / (12980045 + 9858512)
yearly_effect

0.009669567827774757

In [14]:
periodic_effect = yearly_effect * 5
periodic_effect

0.048347839138873784

```xml
<?xml version="1.0" encoding="UTF-8"?><scenario>
    <world>
        <global-technology-database>
            <location-info sector-name="trn_pass_road_LDV_4W" subsector-name="Car">
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <lifetime>25</lifetime>
                        <s-curve-shutdown-decider name="s-curve">
                            <steepness>0.218</steepness>
                            <half-life>11</half-life>
                        </s-curve-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>

            <location-info sector-name="trn_pass_road_LDV_4W" subsector-name="Large Car and Truck">
                
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <lifetime>25</lifetime>
                        <s-curve-shutdown-decider name="s-curve">
                            <steepness>0.23</steepness>
                            <half-life>11</half-life>
                        </s-curve-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>

            <location-info sector-name="trn_freight_road" subsector-name="Medium truck">
                
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <lifetime>20</lifetime>
                        <s-curve-shutdown-decider name="s-curve">
                            <steepness>0.193</steepness>
                            <half-life>10</half-life>
                        </s-curve-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>
        </global-technology-database>
    </world>
</scenario>

```

[S curve shutdown](https://jgcri.github.io/gcam-doc/en_technologies.html)($\text{output}=\frac{1}{1+\exp{s*(t-h)}}$)에 따르면 2015년 LDV-4W Car는

In [15]:
def s_curve(t, s, h):
    return 1 / (1+np.exp(s*(t-h)))

In [16]:
print(s_curve(5, 0.218, 11))
print(s_curve(10, 0.218, 11))
print(s_curve(15, 0.218, 11))
print(s_curve(20, 0.218, 11))

0.7871782910626082
0.5542851826734679
0.2948383140857415
0.12325076464115273


In [17]:
dfRetireCar = pd.DataFrame({
    'bau': [s_curve(5, 0.218, 11), s_curve(10, 0.218, 11), s_curve(15, 0.218, 11), s_curve(20, 0.218, 11)],
    'effect': [yearly_effect*2, yearly_effect*7, yearly_effect*12, yearly_effect*17]
}, index=range(2020, 2040, 5))
dfRetireCar['output-scalar'] = (dfRetireCar['bau'] - dfRetireCar['effect']).clip(lower=0)
dfRetireCar['Type']= 'Car'
dfRetireCar['Scenario']= 'Current'
dfRetireCar

,bau,effect,output-scalar,Type,Scenario
2020,0.787178,0.019339,0.767839,Car,Current
2025,0.554285,0.067687,0.486598,Car,Current
2030,0.294838,0.116035,0.178804,Car,Current
2035,0.123251,0.164383,0.000000,Car,Current


In [18]:
dfRetireLargeCar = pd.DataFrame({
    'bau': [s_curve(5, 0.23, 11), s_curve(10, 0.23, 11), s_curve(15, 0.23, 11), s_curve(20, 0.23, 11)],
    'effect': [yearly_effect*2, yearly_effect*7, yearly_effect*12, yearly_effect*17]
}, index=range(2020, 2040, 5))
dfRetireLargeCar['output-scalar'] = (dfRetireLargeCar['bau'] - dfRetireLargeCar['effect']).clip(lower=0)
dfRetireLargeCar

,bau,effect,output-scalar
2020,0.798991,0.019339,0.779652
2025,0.557248,0.067687,0.489561
2030,0.284958,0.116035,0.168923
2035,0.112047,0.164383,0.000000


In [19]:
dfRetireTruck = pd.DataFrame({
    'bau': [s_curve(5, 0.193, 10), s_curve(10, 0.193, 10), s_curve(15, 0.193, 10), s_curve(20, 0.193, 10)],
    'effect': [yearly_effect*2, yearly_effect*7, yearly_effect*12, yearly_effect*17]
}, index=range(2020, 2040, 5))
dfRetireTruck['output-scalar'] = (dfRetireTruck['bau'] - dfRetireTruck['effect']).clip(lower=0)
dfRetireTruck['Type']= 'Freight Truck'
dfRetireTruck['Scenario']= 'Current'
dfRetireTruck

,bau,effect,output-scalar,Type,Scenario
2020,0.724122,0.019339,0.704783,Freight Truck,Current
2025,0.500000,0.067687,0.432313,Freight Truck,Current
2030,0.275878,0.116035,0.159843,Freight Truck,Current
2035,0.126751,0.164383,0.000000,Freight Truck,Current


In [20]:
dfC = pd.concat([dfRetireCar, dfRetireTruck])
dfC

,bau,effect,output-scalar,Type,Scenario
2020,0.787178,0.019339,0.767839,Car,Current
2025,0.554285,0.067687,0.486598,Car,Current
2030,0.294838,0.116035,0.178804,Car,Current
2035,0.123251,0.164383,0.000000,Car,Current
2020,0.724122,0.019339,0.704783,Freight Truck,Current
2025,0.500000,0.067687,0.432313,Freight Truck,Current
2030,0.275878,0.116035,0.159843,Freight Truck,Current
2035,0.126751,0.164383,0.000000,Freight Truck,Current


```xml


<?xml version="1.0" encoding="UTF-8"?><scenario>
    <world>
        <global-technology-database>
            <location-info sector-name="trn_pass_road_LDV_4W" subsector-name="Car">
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <s-curve-shutdown-decider delete="1" name="s-curve"/>
                        <exogenous-shutdown-decider name="exogenous-shutdown">
                            <output-scalar year="2020">0.767</output-scalar>
                            <output-scalar year="2025">0.486</output-scalar>
                            <output-scalar year="2030">0.178</output-scalar>
                            <output-scalar year="2035">0.001</output-scalar>
                        </exogenous-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>

            <location-info sector-name="trn_pass_road_LDV_4W" subsector-name="Large Car and Truck">
                
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <s-curve-shutdown-decider delete="1" name="s-curve"/>
                        <exogenous-shutdown-decider name="exogenous-shutdown">
                            <output-scalar year="2020">0.767</output-scalar>
                            <output-scalar year="2025">0.486</output-scalar>
                            <output-scalar year="2030">0.178</output-scalar>
                            <output-scalar year="2035">0.001</output-scalar>
                        </exogenous-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>

            <location-info sector-name="trn_freight_road" subsector-name="Medium truck">
                
                <tranTechnology name="Liquids">
                    <period year="2015">
                        <s-curve-shutdown-decider delete="1" name="s-curve"/>
                        <exogenous-shutdown-decider name="exogenous-shutdown">
                            <output-scalar year="2020">0.704</output-scalar>
                            <output-scalar year="2025">0.432</output-scalar>
                            <output-scalar year="2030">0.159</output-scalar>
                            <output-scalar year="2035">0.001</output-scalar>
                        </exogenous-shutdown-decider>
                    </period>
                </tranTechnology>
                
            </location-info>
        </global-technology-database>
    </world>
</scenario>

```